### Incluisón de variables climatológicas

El gusano barrenador habita en zonas concierta humedad y clima cálido, es endeble ante climas fríos, una variable importante para el análisis de infecciones  y lograr el objetivo de crear un tablero con oznas infectadas y de posible infección es la condición climatológica de la zona de estudio. 

Para obtener estas variables podemos utilizar la API de Copernicus (ERA5-Land) vía Google Earth Engine, fue seleccionada por su precisión y desarrollo actual empatando con el desarrollo actual del gusano barrenador. El modelo extraerá las condiciones exactas que existían en ese punto geográfico en la fecha específica del brote. Es importante destacar que las condiciones climáticas a extraer será el promedio de los 30 días previos a la fecha del inicio, esto debido a que el clima de los 30 días previos es aquel que determinó si la mosca sobrevivió, voló y logró reproducirse. 

Se extraerán dos bandas de los satélites:

- temperature_2m: Temperaturas detectadas.
- total_precipitation_sum: La humedad dicta la supervivencia del suelo.

In [ ]:
import sys 
!{sys.executable} -m pip install earthengine-api

In [3]:
import os
import ee
import pandas as pd
import datetime
from tqdm import tqdm

# Configuración de rutas 
directorio_notebooks = os.getcwd()
directorio_raiz = os.path.dirname(directorio_notebooks)

# Construye las rutas exactas independientemente del sistema operativo
ruta_entrada = os.path.join(directorio_raiz, 'data', 'processed', 'Dataset_Gusano_Barrenador_Completo.csv')
ruta_salida = os.path.join(directorio_raiz, 'data', 'processed', 'Dataset_GB_Clima_Final.csv')

# Validación de seguridad antes de gastar recursos
if not os.path.exists(ruta_entrada):
    raise FileNotFoundError(f"No se encontró el archivo base. Verifica que exista en: {ruta_entrada}")

# Autenticación de Google Earth Engine 
try:
    ee.Initialize()
    print("Conectado a Google Earth Engine exitosamente.")
except Exception as e:
    print("Solicitando token de autenticación...")
    ee.Authenticate()
    ee.Initialize()

#  Cargar el dataset
df = pd.read_csv(ruta_entrada)
df['Fecha_Inicio'] = pd.to_datetime(df['Fecha_Inicio'])

temperaturas = []
precipitaciones = []


# Extracción de clima (Ventana biológica de 30 días previos)
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Consultando satélites"):
    try:
        lat = row['Latitud']
        lon = row['Longitud']
        fecha_fin = row['Fecha_Inicio']
        
        # Retroceder 30 días
        fecha_inicio = fecha_fin - pd.Timedelta(days=30)
        
        str_inicio = fecha_inicio.strftime('%Y-%m-%d')
        str_fin = fecha_fin.strftime('%Y-%m-%d')
        
        punto = ee.Geometry.Point([lon, lat])
        
        # Colección ERA5-Land 
        dataset = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
                    .filterBounds(punto) \
                    .filterDate(str_inicio, str_fin)
        
        imagen_promedio = dataset.mean()
        
        datos = imagen_promedio.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=punto,
            scale=9000
        ).getInfo()
        
        # Transformaciones (Kelvin a Celsius, Metros a Milímetros)
        temp_c = (datos.get('temperature_2m') - 273.15) if datos.get('temperature_2m') else None
        precip_mm = (datos.get('total_precipitation_sum') * 1000) if datos.get('total_precipitation_sum') else None
            
        temperaturas.append(temp_c)
        precipitaciones.append(precip_mm)
        
    except Exception as e:
        temperaturas.append(None)
        precipitaciones.append(None)

# Guardado
df['Temperatura_Media_C'] = temperaturas
df['Precipitacion_Acumulada_mm'] = precipitaciones

df_final = df.dropna(subset=['Temperatura_Media_C', 'Precipitacion_Acumulada_mm'])

# Usa la ruta dinámica de salida que creamos al inicio
df_final.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
print(f"\n¡Proceso exitoso! Archivo climático guardado en:\n{ruta_salida}")

Solicitando token de autenticación...


Enter verification code:  4/1ATsMZqD1yBBO-IZAlmU6eSBQhfVUrc2zWB_X6LYnkDCwv5JB7t4osX4ZQeg



Successfully saved authorization token.


Consultando satélites: 100%|██████████| 3304/3304 [2:40:19<00:00,  2.91s/it]  



¡Proceso exitoso! Archivo climático guardado en:
C:\Users\Moises\gusano-barrenador-spatial-risk\data\processed\Dataset_GB_Clima_Final.csv


En el notebook disponible en google colab, se han obtenido la mediana de las variables climáticas sustituyendo al promedio, debido a que condiciones como la temperatura y la precipitación acumulada son sensibles a valores atípicos cuando se maneja el promedio, la mediana dicta de manera significativa el comportamiento general de las condiciones a lo largo del mes, por lo que es una herramienta más precisa cuando se trata de este tipo de variables. 

**Nota Metodológica:** Los datos climáticos descritos en este notebook fueron procesados y extraídos asíncronamente utilizando los servidores de Google Earth Engine. 
[🌐 Ver Script de Extracción en Google Colab](https://colab.research.google.com/drive/1yhUlnHcjPREOiRYKhn1yl2HjsQm3c1Pu#scrollTo=DM5ohryOvbty)